# SAFOD solid-Earth tides: forcing, Thomas Figure 3 analogue, and response models

This notebook is the **presentation layer** for the Sherlock pipeline. PySolid, SPOTL, the transparent tide calculation, and Models A–D are executed by scripts in `scripts/tides/`; this notebook only reads products in `outputs/tides/`.

To regenerate the calculation on Sherlock:

```bash
git pull
bash RUN_ON_SHERLOCK.sh
```

Then select the kernel **SAFOD tides (.venv)** and run this notebook.

The physical hierarchy is

$$
\text{Sun/Moon}
\rightarrow
\boldsymbol{\varepsilon}(t)
\rightarrow
\boldsymbol{\sigma}(t)
\rightarrow
\text{fault traction}
\rightarrow
\Delta v/v.
$$

The tide packages constrain the first part. The larger modeling uncertainty begins with the constitutive and rock-physics steps.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "config.json").exists() and (p / "scripts/tides").exists():
            return p
    raise FileNotFoundError("Could not locate project root.")

ROOT = find_root()
OUT = ROOT / "outputs/tides"
CONFIG = json.loads((ROOT / "config.json").read_text())
print("Project root:", ROOT)
print("Results:", OUT)


## 1. Run status and provenance

A package curve is treated as a package result only when both its numerical output and provenance file exist. Missing SPOTL output is never replaced by an analytic surrogate.


In [ ]:
def load_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None

rows=[]
for name,csv_name,prov_name in [
    ("PySolid","pysolid_tides.csv","pysolid_provenance.json"),
    ("SPOTL ertid","spotl_ertid_tides.csv","spotl_provenance.json"),
    ("analytic degree-2","analytic_degree2_tides.csv","analytic_degree2_provenance.json"),
    ("Models A-D","model_results.csv","model_provenance.json"),
]:
    p=OUT/csv_name
    q=OUT/prov_name
    prov=load_json(q)
    rows.append({
        "product":name, "data":p.exists(), "provenance":q.exists(),
        "hostname":None if prov is None else prov.get("hostname"),
        "created_utc":None if prov is None else prov.get("created_utc"),
        "git_commit":None if prov is None else prov.get("git_commit"),
    })
display(pd.DataFrame(rows))


## 2. Tidal strain forcing

**PySolid** returns solid-Earth-tide displacement. The Sherlock script evaluates a small spatial stencil around SAFOD and differentiates the displacement field to recover the horizontal strain tensor.

**SPOTL `ertid`** can return extensional strain directly. We request strain along $0^\circ$, $45^\circ$, and $90^\circ$ and reconstruct

$$
\varepsilon_{NN},\qquad
\varepsilon_{EE},\qquad
\varepsilon_{NE}.
$$

Thomas et al. (2012) used SPOTL for the Parkfield calculation. For the body tide they argued that the wavelength is sufficiently long that surface strain is not significantly different from strain at 25 km depth. Their ocean-loading treatment was depth dependent; our present comparison is body tide only.


In [ ]:
def read_product(name):
    p=OUT/name
    return None if not p.exists() else pd.read_csv(p,parse_dates=["time_utc"])

pysolid=read_product("pysolid_tides.csv")
spotl=read_product("spotl_ertid_tides.csv")
analytic=read_product("analytic_degree2_tides.csv")

print("PySolid:", "not present" if pysolid is None else len(pysolid))
print("SPOTL:", "not present" if spotl is None else len(spotl))
print("analytic:", "not present" if analytic is None else len(analytic))

if pysolid is not None:
    plt.figure(figsize=(11,5))
    plt.plot(pysolid.time_utc,1e9*(pysolid.areal_strain-pysolid.areal_strain.mean()),
             linewidth=2,label="PySolid")
    if spotl is not None:
        plt.plot(spotl.time_utc,1e9*(spotl.areal_strain-spotl.areal_strain.mean()),
                 label="SPOTL ertid")
    if analytic is not None and "areal_strain" in analytic.columns:
        plt.plot(analytic.time_utc,1e9*(analytic.areal_strain-analytic.areal_strain.mean()),
                 label="transparent degree-2")
    plt.axhline(0,linewidth=.8)
    plt.ylabel("Mean-removed areal strain (nanostrain)")
    plt.xlabel("UTC")
    plt.title("SAFOD body-tide forcing, June 16–17 2026")
    plt.grid(alpha=.25); plt.legend()
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

comparison_path=OUT/"forcing_comparison.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path))


# 3. Model B: strain → elastic stress → fault traction

Thomas et al. (2012) explicitly describe this sequence:

$$
\text{SPOTL strain}
\rightarrow
\text{linear elastic constitutive equation}
\rightarrow
\text{stress resolved onto the SAF}.
$$

They resolve onto a **vertical plane striking N42°W** and plot fault-normal stress (FNS) and right-lateral shear stress (RLSS) in their Figure 3. They state that the solid-Earth-tide stress is largely volumetric, so RLSS is roughly an order of magnitude smaller than FNS.

Thomas et al. do **not** print the constitutive equation or numerical elastic constants in that methods paragraph. To make this step explicit, the present implementation adopts the later Parkfield prescription of van der Elst et al. (2016): **linear elasticity, plane strain, $\nu=0.25$, and $G=30$ GPa**.

For isotropic elasticity,

$$
\sigma_{ij}=2G\varepsilon_{ij}+\lambda\,\varepsilon_{kk}\delta_{ij},
\qquad
\lambda=\frac{2G\nu}{1-2\nu}.
$$

Under the adopted plane-strain closure,

$$
\varepsilon_{DD}=\varepsilon_{ND}=\varepsilon_{ED}=0,
$$

so

$$
\varepsilon_{kk}=\varepsilon_{NN}+\varepsilon_{EE},
$$

and therefore

$$
\sigma_{NN}=2G\varepsilon_{NN}+\lambda(\varepsilon_{NN}+\varepsilon_{EE}),
$$

$$
\sigma_{EE}=2G\varepsilon_{EE}+\lambda(\varepsilon_{NN}+\varepsilon_{EE}),
$$

$$
\sigma_{DD}=\lambda(\varepsilon_{NN}+\varepsilon_{EE}),
\qquad
\sigma_{NE}=2G\varepsilon_{NE}.
$$

This is a **published Parkfield-style closure**, not a claim that plane strain is the exact three-dimensional tidal state at 1 km depth.


## 3.1 Thomas et al. Figure 3 analogue for June 16–17, 2026

Thomas et al. Figure 3 is a 14-day plot of FNS (blue) and RLSS (red) on a vertical N42°W San Andreas plane. Here we reproduce the **same stress components and fault geometry** for our SAFOD experiment window.

This plot is intentionally separate from the primary SAFOD Model B geometry. It answers a validation question:

> Does our strain → elasticity → fault-resolution calculation produce the same qualitative stress hierarchy that Thomas et al. report?

When SPOTL has run, SPOTL is used for this figure because that most closely matches Thomas et al. If SPOTL is absent, the code uses PySolid and labels it accordingly.


In [ ]:
model_path=OUT/"model_results.csv"
models=pd.read_csv(model_path,parse_dates=["time_utc"]) if model_path.exists() else None

if models is None:
    print("No model results yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    forcing = "spotl" if "spotl_thomas_FNS_pa" in models.columns else "pysolid"
    fns_col=f"{forcing}_thomas_FNS_pa"
    rlss_col=f"{forcing}_thomas_RLSS_pa"

    if fns_col not in models.columns:
        print("model_results.csv predates the Thomas-reference outputs. Run git pull and rerun the pipeline.")
    else:
        good=models[["time_utc",fns_col,rlss_col]].dropna()
        plt.figure(figsize=(11,5))
        plt.plot(good.time_utc,good[fns_col]/1000.0,label="FNS")
        plt.plot(good.time_utc,good[rlss_col]/1000.0,label="RLSS")
        plt.axhline(0,linewidth=.8)
        plt.ylabel("Tidally induced stress (kPa)")
        plt.xlabel("UTC")
        plt.title(f"Thomas et al. (2012) Figure 3 analogue — SAFOD window ({forcing})")
        plt.legend(); plt.grid(alpha=.25)
        plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
        plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

        Af=np.max(np.abs(good[fns_col]))
        As=np.max(np.abs(good[rlss_col]))
        print(f"forcing: {forcing}")
        print(f"max |FNS|  = {Af:.2f} Pa")
        print(f"max |RLSS| = {As:.2f} Pa")
        print(f"|FNS|/|RLSS| amplitude ratio = {Af/As:.2f}")


The original Thomas et al. Figure 3 is **not** numerically reproduced here: their plot uses a different date, location, and elastic model details, and includes ocean loading. The comparison we care about is methodological and qualitative. In their calculation, body tides dominate inland and FNS is much larger than RLSS.

## 3.2 Primary SAFOD Model B geometry

For the actual SAFOD application, the same plane-strain stress tensor is resolved onto the configured local fault scenario:

- strike $140^\circ$,
- dip $70^\circ$,
- dip direction $230^\circ$.

For a fault normal $\mathbf n$ and strike direction $\mathbf s$,

$$
\mathbf t=\boldsymbol{\sigma}\mathbf n,
\qquad
\mathrm{FNS}=\mathbf n^T\boldsymbol{\sigma}\mathbf n,
$$

and

$$
\tau_{\mathrm{strike}}=\mathbf s^T\boldsymbol{\sigma}\mathbf n.
$$

FNS is defined positive in tension/unclamping. This SAFOD projection, rather than the Thomas vertical-plane reference, is the stress used by the primary Model B response calculation.


In [ ]:
if models is not None:
    forcing = "spotl" if "spotl_FNS_pa" in models.columns else "pysolid"
    fns=f"{forcing}_FNS_pa"
    shr=f"{forcing}_along_strike_shear_pa"
    if fns in models.columns:
        good=models[["time_utc",fns,shr]].dropna()
        plt.figure(figsize=(11,5))
        plt.plot(good.time_utc,good[fns]/1000,label="SAFOD FNS")
        plt.plot(good.time_utc,good[shr]/1000,label="SAFOD along-strike shear")
        plt.axhline(0,linewidth=.8)
        plt.ylabel("Tidally induced stress (kPa)")
        plt.xlabel("UTC")
        plt.title(f"Model B stress resolved onto SAFOD geometry ({forcing})")
        plt.legend(); plt.grid(alpha=.25)
        plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
        plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()


## 3.3 Stress → $\Delta v/v$ is a separate assumption

The elastic calculation ends at the tidal stress history. Turning that stress into a seismic-velocity perturbation is **not supplied by Thomas et al.**

The present empirical branch applies the SAFOD coefficient from Niu et al. (2008),

$$
S_{\mathrm{Niu}}=2.4\times10^{-7}\ \mathrm{Pa}^{-1},
$$

through

$$
\left(\frac{\Delta v}{v}\right)_B
=
S_{\mathrm{Niu}}\,\Delta\sigma_*.
$$

Using Niu's barometric stress sensitivity for a tidal stress component is an **assumed cross-loading transfer**. It should be kept conceptually separate from the much better constrained tidal strain and elastic-stress calculation.


# 4. Models A–D and AWD detectability

Model A is the Niu 240-Pa amplitude shortcut. Model B is the explicit elastic-stress calculation above followed by the Niu transfer. Model C applies direct strain sensitivities from other sites as context. Model D uses a stress-dependent crack-compliance model.

No branch constitutes a tidal detection.


In [ ]:
if models is not None:
    summary_rows=[]
    for forcing in ["pysolid","spotl"]:
        mapping={
            "A":f"{forcing}_model_A_dv_over_v",
            "B":f"{forcing}_model_B_dv_over_v",
            "C (Takano)":f"{forcing}_model_C_takano_dv_over_v",
            "D (Vs)":f"{forcing}_model_D_dVs_over_Vs",
            "D (Vp)":f"{forcing}_model_D_dVp_over_Vp",
        }
        if all(col in models.columns for col in mapping.values()):
            for name,col in mapping.items():
                amp=np.nanmax(np.abs(models[col]))
                summary_rows.append({
                    "forcing":forcing,
                    "model":name,
                    "max_abs_dv/v":amp,
                    "max_abs_percent":100*amp,
                    "Deep reliable / model":CONFIG["awd_benchmarks"]["deep_outbound_reliable"]/amp,
                })
    summary=pd.DataFrame(summary_rows)
    display(summary.style.format({
        "max_abs_dv/v":"{:.3e}",
        "max_abs_percent":"{:.5f}",
        "Deep reliable / model":"{:.1f}x",
    }))


## 5. Interpretation and limitations

The modeling hierarchy is deliberately explicit:

$$
\boxed{\text{tide packages}}
\rightarrow
\boxed{\boldsymbol{\varepsilon}(t)}
\rightarrow
\boxed{\text{elastic closure}}
\rightarrow
\boxed{\text{fault stress}}
\rightarrow
\boxed{\text{stress-to-velocity model}}.
$$

The PySolid/SPOTL comparison addresses forcing uncertainty. The Thomas-style Figure 3 analogue checks whether the stress construction behaves sensibly. The SAFOD geometry then supplies the site-specific stress scenario. The final stress-to-$\Delta v/v$ step remains the least constrained part.

Important limitations:

- Thomas et al. (2012) do not explicitly state a plane-strain assumption; that closure is taken from the later Parkfield implementation of van der Elst et al. (2016).
- Plane strain is still an approximation to the depth-dependent 3-D tidal stress field.
- Niu's $2.4\times10^{-7}\ \mathrm{Pa}^{-1}$ coefficient is an empirical barometric sensitivity, not a universal tidal coefficient.
- Model D predicts formation-scale $V_P/V_S$, not the AWD guided apparent velocity.
- No tidal response is claimed to have been detected in the AWD experiment.

## References

- Thomas et al. (2012), *JGR Solid Earth*, DOI `10.1029/2011JB009036`.
- van der Elst et al. (2016), *PNAS*, DOI `10.1073/pnas.1524316113`.
- Agnew (2012), SPOTL.
- Niu et al. (2008), *Nature*, DOI `10.1038/nature07111`.
